# Pooling Generation (Qrels)
This notebook generates the evaluation pool by extracting the top results from BM25, Dense, and Hybrid search for a set of test queries.

In [9]:
import pandas as pd
from search_engine import BM25Searcher, DenseSearcher, HybridSearcher

# --- EDIT YOUR TEST QUERIES HERE ---
# Important to decide which are the test queries we want to use
TEST_QUERIES = [
    "The Lord of the Rings",
    "Avengers",
    "A Silent Voice",
    "Inception",
    "Demon slayer",
    "A group of people trying to survive a zombie apocalypse",
    "A detective solving a complex murder mystery",
    "Escaping from a high-security prison",
    "A feel-good family comedy for the holidays",
    "An inspiring sports documentary about underdogs",
    "Movies directed by Christopher Nolan",
    "Starring Leonardo DiCaprio and Tom Hardy",
    "Action movies with strong female lead characters",
    "Korean romantic comedy dramas",
    "Magic school and wizards",
    "Superheroes fighting each other",
    "A Japanese anime about high school teenagers",
    "An animated 3D cartoon movie for little kids",
    "A dark fantasy anime with demons and sword fighting",
    "Anime",
]
# -----------------------------------

In [10]:
# Load the dataframe with embeddings
print("Loading dataframe...")
movies = pd.read_pickle('movies_with_embeddings.pkl')

# Initialize searchers
bm25_searcher = BM25Searcher(movies)
dense_searcher = DenseSearcher(movies)
hybrid_searcher = HybridSearcher(bm25_searcher, dense_searcher)


Loading dataframe...
Tokenizing corpus for BM25...
BM25 Index built successfully.
Loading SentenceTransformer model (paraphrase-multilingual-MiniLM-L12-v2)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dense embeddings loaded successfully.


In [11]:
# Generate Pool
pool_data = []
top_k = 5

for query in TEST_QUERIES:
    print(f"Processing query: '{query}'")
    bm25_res = bm25_searcher.search(query, top_k=top_k)
    dense_res = dense_searcher.search(query, top_k=top_k)
    hybrid_res = hybrid_searcher.search(query, top_k=top_k)
    
    unique_titles = set()
    for res in bm25_res + dense_res + hybrid_res:
        unique_titles.add(res['title'])
        
    for title in unique_titles:
        pool_data.append({
            'query': query,
            'movie_title': title,
            'relevance': '' # To be filled by you!
        })

pool_df = pd.DataFrame(pool_data)
pool_df.to_csv('qrels_to_grade_english.csv', index=False)
print(f"\nPool generated successfully! Please grade the {len(pool_df)} rows in 'qrels_to_grade.csv' (0 to 5 scale).")

Processing query: 'The Lord of the Rings'
Processing query: 'Avengers'
Processing query: 'A Silent Voice'
Processing query: 'Inception'
Processing query: 'Demon slayer'
Processing query: 'A group of people trying to survive a zombie apocalypse'
Processing query: 'A detective solving a complex murder mystery'
Processing query: 'Escaping from a high-security prison'
Processing query: 'A feel-good family comedy for the holidays'
Processing query: 'An inspiring sports documentary about underdogs'
Processing query: 'Movies directed by Christopher Nolan'
Processing query: 'Starring Leonardo DiCaprio and Tom Hardy'
Processing query: 'Action movies with strong female lead characters'
Processing query: 'Korean romantic comedy dramas'
Processing query: 'Magic school and wizards'
Processing query: 'Superheroes fighting each other'
Processing query: 'A Japanese anime about high school teenagers'
Processing query: 'An animated 3D cartoon movie for little kids'
Processing query: 'A dark fantasy anim